# ECG-FM fine-tuning trên PTB-XL ở 250 Hz

Notebook này fine-tune checkpoint ECG-FM có sẵn trên PTB-XL với 5 nhóm chẩn đoán (`NORM`, `MI`, `STTC`, `CD`, `HYP`). Đây là bài toán **multi-label classification**: một ECG có thể có nhiều nhãn dương tính đồng thời.

Pipeline giữ nguyên trọng số pretrained của backbone, chỉ thay classification head `model.proj`; backbone và head được huấn luyện với learning rate riêng. Tín hiệu được đưa về 250 Hz, chuẩn hóa và chia thành các đoạn 5 giây không chồng lấp. Dự đoán các đoạn được gộp trung bình về cấp ECG record trước khi tính metric.

- Train: `strat_fold` 1–8; validation: fold 9; test: fold 10.
- Loss: `BCEWithLogitsLoss` với `pos_weight` theo từng nhãn.
- Optimizer: AdamW; scheduler: warmup + cosine decay; có gradient clipping và early stopping.
- Checkpoint tốt nhất được chọn theo **validation Macro-F1 tại threshold 0.5**. Test chỉ được đánh giá sau khi chọn checkpoint.

**Điều kiện:** `ckpts/mimic_iv_ecg_finetuned.pt` phải là checkpoint pretrained tương thích với kiến trúc mà `fairseq_signals` dựng lên. Nếu checkpoint đó không chứa pretrained weights thực sự, notebook không thể tự tạo ra transfer learning chỉ bằng cách thay classifier.


In [1]:
from pathlib import Path
import ast
import random
from itertools import chain

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from sklearn.metrics import average_precision_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
import wfdb

from ecg_transform.inp import ECGInput, ECGInputSchema
from ecg_transform.sample import ECGMetadata, ECGSample
from ecg_transform.t.common import HandleConstantLeads, LinearResample, ReorderLeads
from ecg_transform.t.cut import SegmentNonoverlapping
from ecg_transform.t.scale import Standardize
from fairseq_signals.models import build_model_from_checkpoint

# -----------------------------------------------------------------------------
# 0. Configuration
# -----------------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PTBXL_DATA_DIR = ROOT / 'data' / 'ptb-xl'
if not PTBXL_DATA_DIR.is_dir():
    raise FileNotFoundError(f'Không tìm thấy thư mục dữ liệu: {PTBXL_DATA_DIR}')
PTBXL_ROOT = next(
    (p for p in PTBXL_DATA_DIR.iterdir() if (p / 'ptbxl_database.csv').is_file()),
    None,
)
if PTBXL_ROOT is None:
    raise FileNotFoundError(
        f'Không tìm thấy ptbxl_database.csv bên trong {PTBXL_DATA_DIR}. '
        'Hãy kiểm tra lại cấu trúc thư mục PTB-XL.'
    )

CHECKPOINT_TEMPLATE = ROOT / 'ckpts' / 'mimic_iv_ecg_finetuned.pt'
if not CHECKPOINT_TEMPLATE.is_file():
    raise FileNotFoundError(f'Không tìm thấy pretrained checkpoint: {CHECKPOINT_TEMPLATE}')

OUTPUT_DIR = ROOT / 'outputs' / 'ptbxl_ecg_fm_250hz_finetuned'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SAMPLE_RATE = 250
SEGMENT_LENGTH = 5 * SAMPLE_RATE
LEADS = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
DIAGNOSTIC_CLASSES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']

BATCH_SIZE = 32                 # số ECG records trong một batch, không phải số segments
NUM_WORKERS = 0
EPOCHS = 30
ENCODER_LR = 1e-5               # learning rate cho các tham số pretrained, trừ model.proj
HEAD_LR = 1e-4                  # learning rate cho classification head mới
WEIGHT_DECAY = 1e-4
WARMUP_RATIO = 0.05
MIN_LR_RATIO = 0.10
MAX_GRAD_NORM = 1.0
THRESHOLD = 0.5
EARLY_STOPPING_PATIENCE = 8

print({
    'device': str(DEVICE),
    'ptbxl_root': str(PTBXL_ROOT),
    'pretrained_checkpoint': str(CHECKPOINT_TEMPLATE),
    'output_dir': str(OUTPUT_DIR),
    'encoder_lr': ENCODER_LR,
    'head_lr': HEAD_LR,
})

{'device': 'cuda', 'ptbxl_root': 'c:\\Users\\admin\\OneDrive - The University of Technology\\Desktop\\ecg-fm\\data\\ptb-xl\\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1', 'pretrained_checkpoint': 'c:\\Users\\admin\\OneDrive - The University of Technology\\Desktop\\ecg-fm\\ckpts\\mimic_iv_ecg_finetuned.pt', 'output_dir': 'c:\\Users\\admin\\OneDrive - The University of Technology\\Desktop\\ecg-fm\\outputs\\ptbxl_ecg_fm_250hz_finetuned', 'encoder_lr': 1e-05, 'head_lr': 0.0001}


In [2]:
# -----------------------------------------------------------------------------
# 1. EDA nhãn + split
# -----------------------------------------------------------------------------
db = pd.read_csv(PTBXL_ROOT / 'ptbxl_database.csv')
scp = pd.read_csv(PTBXL_ROOT / 'scp_statements.csv', index_col=0)

diagnostic_map = scp['diagnostic_class'].dropna().to_dict()

def diagnostic_labels(value):
    codes = ast.literal_eval(value) if isinstance(value, str) else {}
    # Multi-label: một record có thể đồng thời thuộc nhiều diagnostic classes.
    return [
        int(any(diagnostic_map.get(code) == cls for code in codes))
        for cls in DIAGNOSTIC_CLASSES
    ]

db['labels'] = db['scp_codes'].map(diagnostic_labels)
label_matrix = np.asarray(db['labels'].tolist(), dtype=np.float32)
for i, name in enumerate(DIAGNOSTIC_CLASSES):
    db[name] = label_matrix[:, i]

split_names = {**{fold: 'train' for fold in range(1, 9)}, 9: 'valid', 10: 'test'}
db['split'] = db['strat_fold'].map(split_names)
db = db[db['split'].notna()].reset_index(drop=True)

print('Số ECG records theo split:')
print(db['split'].value_counts().sort_index())
print('\nTỷ lệ positive theo lớp và split:')
print(db.groupby('split')[DIAGNOSTIC_CLASSES].mean().round(4))
print('\nSố nhãn dương tính trung bình / record:')
print(db.groupby('split')[DIAGNOSTIC_CLASSES].sum().sum(axis=1) / db.groupby('split').size())

Số ECG records theo split:
split
test      2203
train    17441
valid     2193
Name: count, dtype: int64

Tỷ lệ positive theo lớp và split:
         NORM      MI    STTC      CD     HYP
split                                        
test   0.4376  0.2510  0.2374  0.2261  0.1194
train  0.4362  0.2516  0.2404  0.2243  0.1216
valid  0.4364  0.2481  0.2435  0.2266  0.1236

Số nhãn dương tính trung bình / record:
split
test     1.271448
train    1.274124
valid    1.278158
dtype: float64


In [3]:
# -----------------------------------------------------------------------------
# 2. Dataset + preprocessing + DataLoaders
# -----------------------------------------------------------------------------
class PTBXLFMDataset(Dataset):
    def __init__(self, frame, transforms, schema):
        self.frame = frame.reset_index(drop=True)
        self.transforms = transforms
        self.schema = schema

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        record = str(PTBXL_ROOT / row.filename_hr)
        signal, fields = wfdb.rdsamp(record)
        signal = signal.T.astype(np.float32, copy=False)

        lead_map = {'AVR': 'aVR', 'AVL': 'aVL', 'AVF': 'aVF'}
        lead_names = [lead_map.get(name, name) for name in fields['sig_name']]
        metadata = ECGMetadata(
            sample_rate=int(fields['fs']),
            num_samples=signal.shape[1],
            lead_names=lead_names,
            unit=None,
            input_start=0,
            input_end=signal.shape[1],
        )
        metadata.file = record
        sample = ECGSample(ECGInput(signal, metadata), self.schema, self.transforms)
        target = torch.tensor(row[DIAGNOSTIC_CLASSES].to_numpy(dtype=np.float32))
        return torch.from_numpy(sample.out).float(), record, target


def collate_fn(items):
    # sample.out có dạng [num_segments, n_leads, segment_length].
    sources = torch.cat([item[0] for item in items], dim=0)
    records = list(chain.from_iterable(
        [[item[1]] * item[0].shape[0] for item in items]
    ))
    targets = torch.cat([
        item[2].unsqueeze(0).repeat(item[0].shape[0], 1)
        for item in items
    ], dim=0)
    return sources, records, targets

schema = ECGInputSchema(
    sample_rate=SAMPLE_RATE,
    expected_lead_order=LEADS,
    required_num_samples=SEGMENT_LENGTH,
)
transforms = [
    ReorderLeads(expected_order=LEADS, missing_lead_strategy='raise'),
    LinearResample(desired_sample_rate=SAMPLE_RATE),
    HandleConstantLeads(strategy='zero'),
    Standardize(),
    SegmentNonoverlapping(segment_length=SEGMENT_LENGTH),
]

loaders = {}
for split in ('train', 'valid', 'test'):
    dataset = PTBXLFMDataset(db[db['split'] == split], transforms, schema)
    loaders[split] = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=(split == 'train'),
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == 'cuda'),
        collate_fn=collate_fn,
    )

print('Số ECG records:', {split: len(loader.dataset) for split, loader in loaders.items()})

Số ECG records: {'train': 17441, 'valid': 2193, 'test': 2203}


In [4]:
# -----------------------------------------------------------------------------
# 3. Load pretrained ECG-FM + thay head + optimizer
# -----------------------------------------------------------------------------
# QUAN TRỌNG: KHÔNG reset_parameters() trên toàn bộ model.
# build_model_from_checkpoint phải nạp trọng số pretrained từ checkpoint.
model = build_model_from_checkpoint(checkpoint_path=str(CHECKPOINT_TEMPLATE))

if not hasattr(model, 'proj') or not hasattr(model.proj, 'in_features'):
    raise AttributeError(
        'Không tìm thấy model.proj.in_features. Hãy kiểm tra lại classification head '
        'của kiến trúc ECG-FM trong checkpoint.'
    )

classifier_features = model.proj.in_features
model.proj = torch.nn.Linear(classifier_features, len(DIAGNOSTIC_CLASSES))
torch.nn.init.xavier_uniform_(model.proj.weight)
torch.nn.init.constant_(model.proj.bias, 0.0)
model.to(DEVICE)

# Tính pos_weight từ TRAIN split בלבד, không dùng validation/test.
train_frame = db[db['split'] == 'train']
positive = train_frame[DIAGNOSTIC_CLASSES].sum().to_numpy(dtype=np.float32)
negative = len(train_frame) - positive
pos_weight_np = negative / np.maximum(positive, 1.0)
pos_weight = torch.tensor(pos_weight_np, dtype=torch.float32, device=DEVICE)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Phân nhóm tham số theo chính module model.proj, không phụ thuộc tên encoder.
head_params = list(model.proj.parameters())
head_param_ids = {id(p) for p in head_params}
backbone_params = [p for p in model.parameters() if id(p) not in head_param_ids]
if not head_params or not backbone_params:
    raise RuntimeError('Không chia được tham số backbone và classification head.')

optimizer = torch.optim.AdamW(
    [
        {'params': backbone_params, 'lr': ENCODER_LR, 'name': 'pretrained_backbone'},
        {'params': head_params, 'lr': HEAD_LR, 'name': 'classification_head'},
    ],
    weight_decay=WEIGHT_DECAY,
)

# Warmup ngắn, sau đó cosine decay đến MIN_LR_RATIO * learning rate ban đầu.
steps_per_epoch = len(loaders['train'])
total_steps = max(steps_per_epoch * EPOCHS, 1)
warmup_steps = int(total_steps * WARMUP_RATIO)

def lr_lambda(step):
    if warmup_steps > 0 and step < warmup_steps:
        return float(step + 1) / float(warmup_steps)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    progress = min(max(progress, 0.0), 1.0)
    cosine = 0.5 * (1.0 + np.cos(np.pi * progress))
    return MIN_LR_RATIO + (1.0 - MIN_LR_RATIO) * float(cosine)

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

print('Backbone params:', sum(p.numel() for p in backbone_params))
print('Head params:', sum(p.numel() for p in head_params))
print('pos_weight:', dict(zip(DIAGNOSTIC_CLASSES, pos_weight.detach().cpu().tolist())))
print('total optimizer steps:', total_steps, '| warmup steps:', warmup_steps)

Backbone params: 90373248
Head params: 3845
pos_weight: {'NORM': 1.2927566766738892, 'MI': 2.9737980365753174, 'STTC': 3.1595516204833984, 'CD': 3.4583332538604736, 'HYP': 7.223008155822754}
total optimizer steps: 16380 | warmup steps: 819


In [5]:
# -----------------------------------------------------------------------------
# 4. Metrics + evaluation ở cấp ECG record
# -----------------------------------------------------------------------------
def record_level(values, records, reducer='mean'):
    frame = pd.DataFrame(values, index=pd.Index(records, name='record'), columns=DIAGNOSTIC_CLASSES)
    return frame.groupby(level=0).agg(reducer).sort_index()


def metrics(y_true, y_score, threshold=THRESHOLD):
    y_true = np.asarray(y_true, dtype=np.int32)
    y_score = np.asarray(y_score, dtype=np.float64)
    y_pred = (y_score >= threshold).astype(np.int32)

    auroc_values, auprc_values = [], []
    f1_values, recall_values, specificity_values = [], [], []
    per_class = {}

    for i, class_name in enumerate(DIAGNOSTIC_CLASSES):
        truth = y_true[:, i]
        pred = y_pred[:, i]
        score = y_score[:, i]

        # AUROC/AP không xác định nếu validation chỉ có một trạng thái của nhãn.
        if np.unique(truth).size == 2:
            auroc_values.append(roc_auc_score(truth, score))
            auprc_values.append(average_precision_score(truth, score))

        tp = np.sum((truth == 1) & (pred == 1))
        tn = np.sum((truth == 0) & (pred == 0))
        fp = np.sum((truth == 0) & (pred == 1))
        fn = np.sum((truth == 1) & (pred == 0))

        f1_i = 2 * tp / max(2 * tp + fp + fn, 1)
        recall_i = tp / max(tp + fn, 1)
        specificity_i = tn / max(tn + fp, 1)
        f1_values.append(f1_i)
        recall_values.append(recall_i)
        specificity_values.append(specificity_i)

        per_class[class_name] = {
            'positive_rate_true': float(truth.mean()),
            'positive_rate_pred': float(pred.mean()),
            'F1': float(f1_i),
            'Recall': float(recall_i),
            'Specificity': float(specificity_i),
        }

    result = {
        'Macro-AUROC': float(np.mean(auroc_values)) if auroc_values else float('nan'),
        # sklearn average_precision_score là Average Precision (AP), thường được báo cáo
        # như AUPRC/AP trong các bài toán phân loại mất cân bằng.
        'Macro-AUPRC': float(np.mean(auprc_values)) if auprc_values else float('nan'),
        'Macro-F1': float(np.mean(f1_values)),
        'Macro-Recall': float(np.mean(recall_values)),
        'Macro-Specificity': float(np.mean(specificity_values)),
    }
    return result, pd.DataFrame.from_dict(per_class, orient='index')


def evaluate(split):
    model.eval()
    loss_sum = 0.0
    segment_count = 0
    scores, targets, records = [], [], []

    with torch.inference_mode():
        for source, batch_records, target in loaders[split]:
            source_device = source.to(DEVICE, non_blocking=True)
            target_device = target.to(DEVICE, non_blocking=True)
            logits = model(source=source_device)['out']
            loss = criterion(logits, target_device)

            n_segments = target.shape[0]
            loss_sum += loss.item() * n_segments
            segment_count += n_segments
            scores.append(torch.sigmoid(logits).cpu().numpy())
            targets.append(target.cpu().numpy())
            records.extend(batch_records)

    # Scores: mean probability over segments. Labels: identical per ECG, max is safe.
    score_by_record = record_level(np.concatenate(scores), records, reducer='mean')
    target_by_record = record_level(np.concatenate(targets), records, reducer='max')
    metric_values, per_class_df = metrics(
        target_by_record.to_numpy(),
        score_by_record.to_numpy(),
        threshold=THRESHOLD,
    )
    mean_loss = loss_sum / max(segment_count, 1)
    return mean_loss, metric_values, target_by_record, score_by_record, per_class_df

In [6]:
# -----------------------------------------------------------------------------
# 5. Training loop + checkpointing + test evaluation
# -----------------------------------------------------------------------------
history = []
best_value = -np.inf
best_epoch = None
no_improve_epochs = 0
interrupted = False
best_path = OUTPUT_DIR / 'best_checkpoint.pt'
last_path = OUTPUT_DIR / 'last_checkpoint.pt'
history_path = OUTPUT_DIR / 'training_history.csv'
plot_path = OUTPUT_DIR / 'training_metrics.png'

def checkpoint_payload(epoch, interrupted=False):
    return {
        'epoch': int(epoch),
        'completed_epochs': len(history),
        'interrupted': bool(interrupted),
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'history': history,
        'labels': DIAGNOSTIC_CLASSES,
        'sample_rate': SAMPLE_RATE,
        'threshold': THRESHOLD,
        'best_metric': 'Macro-F1',
        'best_metric_value': float(best_value),
    }

def save_history_and_plot():
    history_df = pd.DataFrame(history)
    history_df.to_csv(history_path, index=False)
    if history_df.empty:
        return
    metric_specs = [
        ('train_loss', 'Train loss'),
        ('valid_loss', 'Validation loss'),
        ('valid_Macro-AUROC', 'Validation Macro-AUROC'),
        ('valid_Macro-AUPRC', 'Validation Macro-AUPRC'),
        ('valid_Macro-F1', 'Validation Macro-F1'),
        ('valid_Macro-Recall', 'Validation Macro-Recall'),
        ('valid_Macro-Specificity', 'Validation Macro-Specificity'),
    ]
    fig, axes = plt.subplots(4, 2, figsize=(14, 16), squeeze=False)
    axes = axes.ravel()
    for ax, (column, title) in zip(axes, metric_specs):
        if column in history_df:
            ax.plot(history_df['epoch'], history_df[column], marker='o', linewidth=1.5)
            ax.set_title(title)
            ax.set_xlabel('Epoch')
            ax.set_ylabel(column)
            ax.grid(True, alpha=0.3)
    for ax in axes[len(metric_specs):]:
        ax.axis('off')
    fig.suptitle('ECG-FM fine-tuning metrics', fontsize=16)
    fig.tight_layout()
    fig.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close(fig)

try:
    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss_sum = 0.0
        train_segment_count = 0

        for source, _, target in loaders['train']:
            source = source.to(DEVICE, non_blocking=True)
            target = target.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(source=source)['out']
            loss = criterion(logits, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            n_segments = target.shape[0]
            train_loss_sum += loss.item() * n_segments
            train_segment_count += n_segments

        train_loss = train_loss_sum / max(train_segment_count, 1)
        valid_loss, valid_metrics, valid_targets, valid_scores, valid_per_class = evaluate('valid')
        row = {
            'epoch': epoch,
            'train_loss': float(train_loss),
            'valid_loss': float(valid_loss),
            **{f'valid_{key}': value for key, value in valid_metrics.items()},
            'encoder_lr': optimizer.param_groups[0]['lr'],
            'head_lr': optimizer.param_groups[1]['lr'],
        }
        history.append(row)

        current_value = valid_metrics['Macro-F1']
        if np.isfinite(current_value) and current_value > best_value:
            best_value = current_value
            best_epoch = epoch
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1

        torch.save(checkpoint_payload(epoch), last_path)
        if best_epoch == epoch:
            torch.save(checkpoint_payload(epoch), best_path)
        save_history_and_plot()

        print(row)
        print('Validation theo từng lớp:')
        display(valid_per_class.round(4))
        if no_improve_epochs >= EARLY_STOPPING_PATIENCE:
            print(f'Early stopping tại epoch {epoch}; best epoch={best_epoch}, best validation Macro-F1={best_value:.4f}')
            break
except KeyboardInterrupt:
    interrupted = True
    print('Đã ngắt training. Đang lưu checkpoint và biểu đồ của các epoch đã hoàn tất...')
finally:
    # Luôn lưu lịch sử/biểu đồ; last_checkpoint có thể là trạng thái giữa epoch.
    current_epoch = locals().get('epoch', 0)
    torch.save(checkpoint_payload(current_epoch, interrupted=interrupted), last_path)
    save_history_and_plot()
    print('Đã lưu:', last_path, history_path, plot_path)

if interrupted:
    print(f'Đã lưu {len(history)} epoch hoàn tất; có thể tiếp tục chạy cell này sau.')
else:
    if not best_path.is_file():
        raise RuntimeError('Không tạo được best_checkpoint.pt; hãy kiểm tra validation metrics.')

    # Chỉ đánh giá test sau khi hoàn tất model selection.
    try:
        best = torch.load(best_path, map_location=DEVICE, weights_only=False)
    except TypeError:
        best = torch.load(best_path, map_location=DEVICE)
    model.load_state_dict(best['model_state_dict'])
    test_loss, test_metrics, test_targets, test_scores, test_per_class = evaluate('test')
    print(f'BEST EPOCH: {best["epoch"]}')
    print(f'BEST VALIDATION MACRO-F1: {best["best_metric_value"]:.4f}')
    print('TEST LOSS:', test_loss)
    print(pd.Series(test_metrics, name='test'))
    print('\nTest theo từng lớp:')
    display(test_per_class.round(4))
    test_scores.to_csv(OUTPUT_DIR / 'test_scores.csv')
    test_targets.to_csv(OUTPUT_DIR / 'test_targets.csv')
    test_per_class.to_csv(OUTPUT_DIR / 'test_per_class_metrics.csv')
    assert best_path.is_file() and last_path.is_file()
    print('Artifacts saved to:', OUTPUT_DIR)

{'epoch': 1, 'train_loss': 1.9944032973792327, 'valid_loss': 0.6846652966463235, 'valid_Macro-AUROC': 0.8929266979123952, 'valid_Macro-AUPRC': 0.7405346106767836, 'valid_Macro-F1': 0.6468864525188627, 'valid_Macro-Recall': 0.828540315979421, 'valid_Macro-Specificity': 0.7685984689828876, 'encoder_lr': 6.678876678876679e-06, 'head_lr': 6.678876678876678e-05}
Validation theo từng lớp:


,positive_rate_true,positive_rate_pred,F1,Recall,Specificity
NORM,0.4364,0.5563,0.8342,0.9488,0.7476
MI,0.2481,0.2959,0.6907,0.7574,0.8563
STTC,0.2435,0.4710,0.6509,0.9551,0.6847
CD,0.2266,0.2107,0.6903,0.6660,0.9228
HYP,0.1236,0.4236,0.3683,0.8155,0.6316


{'epoch': 2, 'train_loss': 0.6295871606462832, 'valid_loss': 0.7087085702840019, 'valid_Macro-AUROC': 0.9037380635166302, 'valid_Macro-AUPRC': 0.7656260936850099, 'valid_Macro-F1': 0.6863978061619864, 'valid_Macro-Recall': 0.7198011580733847, 'valid_Macro-Specificity': 0.8808491986519682, 'encoder_lr': 9.993166823949924e-06, 'head_lr': 9.993166823949923e-05}
Validation theo từng lớp:


,positive_rate_true,positive_rate_pred,F1,Recall,Specificity
NORM,0.4364,0.5312,0.8473,0.9394,0.7848
MI,0.2481,0.2891,0.7301,0.7904,0.8763
STTC,0.2435,0.1984,0.7038,0.6386,0.9433
CD,0.2266,0.1555,0.7112,0.5996,0.9746
HYP,0.1236,0.2312,0.4396,0.6310,0.8252


{'epoch': 3, 'train_loss': 0.5656096879553592, 'valid_loss': 0.6490257671310975, 'valid_Macro-AUROC': 0.91143367658636, 'valid_Macro-AUPRC': 0.7724377755779082, 'valid_Macro-F1': 0.7004308330066215, 'valid_Macro-Recall': 0.7943332269067216, 'valid_Macro-Specificity': 0.8515738393552439, 'encoder_lr': 9.938625865312252e-06, 'head_lr': 9.938625865312251e-05}
Validation theo từng lớp:


,positive_rate_true,positive_rate_pred,F1,Recall,Specificity
NORM,0.4364,0.5399,0.8426,0.9425,0.7718
MI,0.2481,0.2426,0.7379,0.7298,0.9181
STTC,0.2435,0.3534,0.7242,0.8876,0.8186
CD,0.2266,0.1888,0.7311,0.6700,0.9522
HYP,0.1236,0.2695,0.4664,0.7417,0.7971


... (7 outputs left out)

,positive_rate_true,positive_rate_pred,F1,Recall,Specificity
NORM,0.4364,0.4528,0.8441,0.8600,0.8625
MI,0.2481,0.3311,0.7165,0.8364,0.8357
STTC,0.2435,0.3634,0.7153,0.8914,0.8065
CD,0.2266,0.2531,0.7243,0.7666,0.8974
HYP,0.1236,0.2057,0.5014,0.6679,0.8595


{'epoch': 8, 'train_loss': 0.3038766315962687, 'valid_loss': 0.8705712481823568, 'valid_Macro-AUROC': 0.9101283014867997, 'valid_Macro-AUPRC': 0.7751784365274135, 'valid_Macro-F1': 0.6987819503607846, 'valid_Macro-Recall': 0.784482433935343, 'valid_Macro-Specificity': 0.8622423859962007, 'encoder_lr': 8.893479743996034e-06, 'head_lr': 8.893479743996034e-05}
Validation theo từng lớp:


,positive_rate_true,positive_rate_pred,F1,Recall,Specificity
NORM,0.4364,0.4843,0.8509,0.8976,0.8358
MI,0.2481,0.2139,0.7206,0.6710,0.9369
STTC,0.2435,0.3420,0.7290,0.8764,0.8300
CD,0.2266,0.2020,0.7468,0.7062,0.9458
HYP,0.1236,0.3032,0.4466,0.7712,0.7627


Đã ngắt training. Đang lưu checkpoint và biểu đồ của các epoch đã hoàn tất...
Đã lưu: c:\Users\admin\OneDrive - The University of Technology\Desktop\ecg-fm\outputs\ptbxl_ecg_fm_250hz_finetuned\last_checkpoint.pt c:\Users\admin\OneDrive - The University of Technology\Desktop\ecg-fm\outputs\ptbxl_ecg_fm_250hz_finetuned\training_history.csv c:\Users\admin\OneDrive - The University of Technology\Desktop\ecg-fm\outputs\ptbxl_ecg_fm_250hz_finetuned\training_metrics.png
Đã lưu 8 epoch hoàn tất; có thể tiếp tục chạy cell này sau.
